In [1]:
import pandas as pd
import numpy as np
from scipy.stats import poisson
from sklearn.metrics import roc_auc_score, accuracy_score, log_loss
from sklearn.preprocessing import label_binarize
import os

# ==========================================
# 1. SETUP & LOAD
# ==========================================
FILENAME = 'matches.xlsx'

try:
    print(f"Loading {FILENAME}...")
    # Add a fallback just in case you are using the CSV version
    if os.path.exists(FILENAME):
        df = pd.read_excel(FILENAME)
    elif os.path.exists('matches.csv'):
        df = pd.read_csv('matches.csv')
    else:
        raise FileNotFoundError
        
    df.columns = df.columns.str.strip()
    
    # --- CLEANING STEP ---
    # 1. Remove invisible spaces from team names
    df['HomeTeam'] = df['HomeTeam'].astype(str).str.strip()
    df['AwayTeam'] = df['AwayTeam'].astype(str).str.strip()
    
    # 2. Fix known typos automatically
    typo_fix = {
        'Totttenham': 'Tottenham',
        'West Han': 'West Ham',
        'Man Utd': 'Man United', 
        'Manchester United': 'Man United'
    }
    df['HomeTeam'] = df['HomeTeam'].replace(typo_fix)
    df['AwayTeam'] = df['AwayTeam'].replace(typo_fix)
    # ---------------------------

    # Required columns adapted for matches.xlsx
    required = ['Date', 'League', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'Home_xG', 'Away_xG']
    if not all(col in df.columns for col in required):
        print(f"❌ Error: Missing columns. Needed: {required}")
        exit()

    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date').reset_index(drop=True)
    
    # Drop rows that don't have results yet (future games)
    df = df.dropna(subset=['FTHG', 'FTAG'])
    print(f"✅ Loaded {len(df)} matches. Running Efficiency Test...")

except FileNotFoundError:
    print(f"❌ Error: '{FILENAME}' not found.")
    exit()

# ==========================================
# 2. THE ENGINE (With Efficiency)
# ==========================================
predictions = []
actuals = []
leagues_tracked = [] 
min_games = 6

for i in range(len(df)):
    # A. Current Match
    row = df.iloc[i]
    date, home, away, league = row['Date'], row['HomeTeam'], row['AwayTeam'], row['League']

    # B. History (Strictly Past)
    history = df[df['Date'] < date]
    league_hist = history[history['League'] == league]
    
    if len(league_hist) < 20: continue 

    # C. Calculate Baselines
    avg_h_xg = league_hist['Home_xG'].mean()
    avg_a_xg = league_hist['Away_xG'].mean()
    
    # Specific Home/Away records
    h_home_rec = league_hist[league_hist['HomeTeam'] == home]
    a_away_rec = league_hist[league_hist['AwayTeam'] == away]
    
    if len(h_home_rec) < min_games or len(a_away_rec) < min_games:
        continue

    # --- 1. CALCULATE xG STRENGTH ---
    h_att = h_home_rec['Home_xG'].mean() / avg_h_xg
    h_def = h_home_rec['Away_xG'].mean() / avg_a_xg # Away_xG here is what Home conceded
    
    a_att = a_away_rec['Away_xG'].mean() / avg_a_xg
    a_def = a_away_rec['Home_xG'].mean() / avg_h_xg # Home_xG here is what Away conceded
    
    raw_exp_h = h_att * a_def * avg_h_xg
    raw_exp_a = a_att * h_def * avg_a_xg

    # --- 2. CALCULATE EFFICIENCY ---
    # Home Team Efficiency
    h_goals = h_home_rec['FTHG'].sum()
    h_xg_total = h_home_rec['Home_xG'].sum()
    h_eff = h_goals / h_xg_total if h_xg_total > 0 else 1.0
    
    # Away Team Efficiency
    a_goals = a_away_rec['FTAG'].sum()
    a_xg_total = a_away_rec['Away_xG'].sum()
    a_eff = a_goals / a_xg_total if a_xg_total > 0 else 1.0
    
    # SAFETY CLAMP
    h_eff = max(0.85, min(h_eff, 1.15))
    a_eff = max(0.85, min(a_eff, 1.15))

    # --- 3. APPLY EFFICIENCY ---
    final_exp_h = raw_exp_h * h_eff
    final_exp_a = raw_exp_a * a_eff
    
    # --- 4. PREDICT ---
    max_g = 10
    grid = np.outer(poisson.pmf(np.arange(max_g), final_exp_h), 
                    poisson.pmf(np.arange(max_g), final_exp_a))
    
    p_away = np.sum(np.triu(grid, 1))
    p_draw = np.sum(np.diag(grid))
    p_home = np.sum(np.tril(grid, -1))
    
    predictions.append([p_away, p_draw, p_home])
    leagues_tracked.append(league) 
    
    # Actual Result
    if row['FTHG'] > row['FTAG']: res = 2
    elif row['FTHG'] == row['FTAG']: res = 1
    else: res = 0
    actuals.append(res)

# ==========================================
# 3. RESULTS (Breakdown)
# ==========================================
if len(actuals) < 10:
    print("Not enough data.")
else:
    # A. OVERALL STATS
    y_test = label_binarize(actuals, classes=[0, 1, 2])
    y_pred = np.array(predictions)
    
    roc = roc_auc_score(y_test, y_pred, multi_class='ovr')
    acc = accuracy_score(actuals, np.argmax(y_pred, axis=1))
    loss = log_loss(actuals, y_pred)
    
    print(f"\n" + "="*50)
    print(f"OVERALL RESULTS ({len(actuals)} Matches)")
    print(f"="*50)
    print(f"ROC AUC Score:   {roc:.4f}")
    print(f"Accuracy:        {acc:.1%}")
    print(f"Log Loss:        {loss:.4f}")
    print(f"="*50)
    
    # B. LEAGUE BY LEAGUE BREAKDOWN
    df_results = pd.DataFrame({
        'Actual': actuals,
        'League': leagues_tracked
    })
    
    unique_leagues = df_results['League'].unique()
    
    print(f"\nLEAGUE PERFORMANCE BREAKDOWN")
    print(f"{'League':<20} | {'Matches':<8} | {'ROC':<8} | {'Acc':<8}")
    print("-" * 55)
    
    for lg in unique_leagues:
        indices = df_results.index[df_results['League'] == lg].tolist()
        if len(indices) < 5: continue 
        
        y_true_lg = np.array([actuals[i] for i in indices])
        y_pred_lg = np.array([predictions[i] for i in indices])
        
        try:
            if len(np.unique(y_true_lg)) > 1:
                y_test_lg = label_binarize(y_true_lg, classes=[0, 1, 2])
                roc_lg = roc_auc_score(y_test_lg, y_pred_lg, multi_class='ovr')
                roc_str = f"{roc_lg:.3f}"
            else:
                roc_str = "N/A"
            
            acc_lg = accuracy_score(y_true_lg, np.argmax(y_pred_lg, axis=1))
            print(f"{lg:<20} | {len(indices):<8} | {roc_str:<8} | {acc_lg:.1%}")
            
        except Exception as e:
            print(f"{lg:<20} | Error calculating stats")

Loading matches.xlsx...
✅ Loaded 1247 matches. Running Efficiency Test...

OVERALL RESULTS (762 Matches)
ROC AUC Score:   0.6210
Accuracy:        48.8%
Log Loss:        1.0209

LEAGUE PERFORMANCE BREAKDOWN
League               | Matches  | ROC      | Acc     
-------------------------------------------------------
La Liga              | 183      | 0.617    | 48.6%
Bundesliga           | 163      | 0.668    | 49.1%
Serie A              | 210      | 0.619    | 52.4%
EPL                  | 206      | 0.593    | 45.1%


C:\Users\jordi\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:259: UserWarning: The y_prob values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
